# Lab 3 — Active Learning I: Uncertainty Sampling

In the previous lab, we trained a baseline. We assumed that labeled training examples were already available. In practice, however, obtaining labels requires human effort, and we may not be able to annotate every message.

**In this lab we will experiment with active learning:** Given a limited annotation budget, can we use the model's uncertainty to decide which messages should be labeled next?

In this notebook, we will:

1. Measure uncertainty in the model's predictions.
2. Select examples for annotation using different acquisition strategies.
3. Simulate an active-learning process with a limited labeling budget.
4. Compare active learning against random sampling.
5. Evaluate whether targeted annotation improves label efficiency.

### The Active-Learning Formulation

We will continue using our customer-support intent-classification task.

Instead of training on all available labeled examples, we will start with a small labeled dataset and treat the remaining training examples as an **unlabeled pool**.

Our active-learning system will repeatedly:

1. Train a classifier on the currently labeled examples.
2. Estimate uncertainty for messages in the unlabeled pool.
3. Select a batch of messages to query.
4. Obtain their labels and add them to the training data.
5. Retrain and evaluate the classifier.

**Important:** We already have reference labels for the dataset. For this experiment, we will hide those labels from the learner and reveal them only when an example is selected. This **simulates the annotation process** without requiring actual human labeling during the lab.

## 1. Data Preparation

For our **intent classification** task, we continue using the **BANKING77** dataset, available on Hugging Face as `PolyAI/banking77`.

### 1.1 Loading the dataset

You should already have downloaded the dataset already in the previous lab. If not, download the dataset beforehand using the `download_dataset.py` script:

````bash
python scripts/download_dataset.py "PolyAI/banking77" "data/PolyAI_banking77"
````

In [ ]:
from datasets import load_dataset, load_from_disk

#ds = load_dataset("PolyAI/banking77"
ds = load_from_disk("../../data/PolyAI_banking77")

ds

### 1.2 Preparing the dataset

For this lab, we continue with the same setting as in our baseline: we focus on **nine related intents** (out of 77) involving account access, transfers, card payments, and cash withdrawals.

We will also create a validation split, as the original dataset provides predefined training and test sets, but no validation set.

In [ ]:
import pandas as pd

# Labels are stored as integers; retrieve their names
label_names = ds["train"].features["label"].names

# Convert the training set to Pandas 
df_train = ds["train"].to_pandas()
df_train["intent"] = df_train["label"].map(lambda i: label_names[i])

# Convert the test set to Pandas as well
df_test = ds["test"].to_pandas()
df_test["intent"] = df_test["label"].map(lambda i: label_names[i])

# let's show a sample
display(df_train.sample(5, random_state=42))

In [ ]:
from sklearn.model_selection import train_test_split

# Original BANKING77 label -> course label
source_to_intent = {
    "passcode_forgotten": "account_access_passcode",
    "pending_transfer": "bank_transfer_pending",
    "transfer_not_received_by_recipient": "bank_transfer_not_received",
    "pending_card_payment": "card_payment_pending",
    "declined_card_payment": "card_payment_declined",
    "transaction_charged_twice": "card_payment_duplicated",
    "pending_cash_withdrawal": "cash_withdrawal_pending",
    "cash_withdrawal_not_recognised": "cash_withdrawal_unrecognized",
    "card_acceptance" : "card_acceptance"
}

# Original BANKING77 intents we want to keep
target_intents = list(source_to_intent.keys())

# Filter the datasets
df_train = df_train[df_train["intent"].isin(target_intents)].copy()
df_test = df_test[df_test["intent"].isin(target_intents)].copy()

# Rename the intents
df_train["intent"] = df_train["intent"].map(source_to_intent)
df_test["intent"] = df_test["intent"].map(source_to_intent)


SEED = 42

df_train_candidates, df_val = train_test_split(
    df_train,
    test_size=0.20,
    stratify=df_train["intent"],
    random_state=SEED
)

# Keep the official test set separate
print("Training candidates:", len(df_train_candidates))
print("Validation:", len(df_val))
print("Test (reserved):", len(df_test))

# Display class distribution
display(df_train["intent"].value_counts())


### 1.3 Creating the Active-Learning Pool

In active learning, we distinguish between two sets:

- **Labeled set ($L_0$)**: examples whose labels are available for training.
- **Unlabeled pool ($U_0$)**: examples whose labels are not yet available to the learner.

We will start with only **10 labeled examples per intent** (90 in total). All remaining training candidates will form our unlabeled pool.

Since BANKING77 is already labeled, we will **simulate annotation** by hiding the labels in the pool and revealing them only when the acquisition strategy selects an example.

In [ ]:

# Initial labeled examples per intent
N_INITIAL_PER_CLASS = 10

# Sample the initial labeled set
df_labeled = (
    df_train_candidates
    .groupby("intent", group_keys=False)
    .sample(n=N_INITIAL_PER_CLASS, random_state=SEED)
    .copy()
)

# All remaining examples form the unlabeled pool
df_pool = df_train_candidates.drop(df_labeled.index).copy()

print("Initial labeled set:", len(df_labeled))
print("Unlabeled pool:", len(df_pool))
print("Validation set:", len(df_val))
print("Reserved test set:", len(df_test))

display(df_labeled["intent"].value_counts())


Important: `df_pool` still contains its reference labels internally. That's useful for our simulated oracle, but acquisition functions must only access the message text and model predictions—not the intent column.

In [ ]:
# Inspect examples without revealing their labels
display(
    df_pool[["text"]].sample(5, random_state=SEED)
)


### 1.4 Defining the Annotation Budget

We cannot label the entire pool. Instead, we have a fixed budget of additional annotations.

For this experiment:

- **Initial labeled set:** 90 examples
- **Query batch size:** 20 examples per round
- **Annotation budget:** 200 additional labels
- **Active-learning rounds:** 10

In each round, the acquisition strategy selects 20 examples from the unlabeled pool. Their labels are revealed, and the classifier is retrained using the expanded labeled set.

We will compare different acquisition strategies under the same initial conditions and annotation budget.


In [ ]:

# Active-learning configuration
BATCH_SIZE = 20
ANNOTATION_BUDGET = 200

N_ROUNDS = ANNOTATION_BUDGET // BATCH_SIZE

print("Initial labeled examples:", len(df_labeled))
print("Additional annotation budget:", ANNOTATION_BUDGET)
print("Query batch size:", BATCH_SIZE)
print("Number of rounds:", N_ROUNDS)
print("Final labeled set size:", len(df_labeled) + ANNOTATION_BUDGET)

assert ANNOTATION_BUDGET <= len(df_pool)
assert ANNOTATION_BUDGET % BATCH_SIZE == 0


## 2. Training the Initial Classifier

### 2.1 Training the Initial Model

We begin by training a classifier using only our initial labeled set $(L_0)$.

This model, denoted $(f_0)$, has seen just 90 labeled examples—10 from each intent. Its predictions will help us identify which examples in the unlabeled pool should be queried next.

We will use the same text-classification approach as in the baseline lab: **TF-IDF features and Logistic Regression**.

The model will provide both predicted intent labels and class probabilities. These probabilities will later be used to calculate uncertainty scores.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Define the classifier
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ))
])

# Train only on the initial labeled set
model.fit(df_labeled["text"], df_labeled["intent"])

print("Initial classifier trained on", len(df_labeled), "examples.")


### 2.2 Evaluating the Initial Model

How well can our classifier perform with only 90 labeled examples?

We evaluate $(f_0)$ on the validation set, which is kept separate from the active-learning pool.

We will use two metrics:

- **Accuracy:** proportion of correctly classified messages.
- **Macro-F1:** average F1-score across all intents, giving each class equal weight.

This gives us the starting point for our learning curves. Later, we will measure whether acquiring additional labels improves performance.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Predict on the validation set
y_val_pred = model.predict(df_val["text"])

# Calculate evaluation metrics
initial_accuracy = accuracy_score(df_val["intent"], y_val_pred)
initial_f1 = f1_score(df_val["intent"], y_val_pred, average="macro")

print(f"Initial accuracy: {initial_accuracy:.3f}")
print(f"Initial macro-F1: {initial_f1:.3f}")


### 2.3 Inspecting Predicted Probabilities

The classifier predicts a probability distribution over the nine intents for each message.

Some predictions may strongly favor one intent, while others may distribute probability across several plausible intents.

Consider two examples:

- **Confident prediction:** one intent receives most of the predicted probability.
- **Uncertain prediction:** several intents receive similar probabilities.

In active learning, we can use these probabilities to estimate which messages might benefit most from human annotation.

Let's inspect a few predictions from our unlabeled pool.


In [ ]:
# Select a few messages from the unlabeled pool
sample_pool = df_pool[["text"]].sample(5, random_state=SEED)

# Predict classes and probabilities
predicted_intents = model.predict(sample_pool["text"])
probabilities = model.predict_proba(sample_pool["text"])

# Build a table for inspection
df_predictions = sample_pool.copy()
df_predictions["predicted_intent"] = predicted_intents
df_predictions["confidence"] = probabilities.max(axis=1)

display(df_predictions)

In [ ]:

# Select one message
example_text = sample_pool["text"].iloc[0]

# Obtain its predicted class probabilities
probs = model.predict_proba([example_text])[0]

# Create a sorted probability table
df_probs = pd.DataFrame({
    "intent": model.classes_,
    "probability": probs
}).sort_values("probability", ascending=False)

print("Message:", example_text)
display(df_probs)


## 3. Uncertainty Sampling

### 3.1 Measuring Predictive Uncertainty

Our classifier produces a probability distribution over the possible intents for each message.

**Uncertainty sampling** uses these probabilities to identify examples for which the model is least certain about its prediction.

We will compare three acquisition functions:

- **Least confidence:** How low is the probability of the most likely class?
- **Margin sampling:** How close are the two most likely classes?
- **Entropy sampling:** How spread out is the entire probability distribution?

For all three methods, a **higher acquisition score means higher uncertainty** and therefore higher priority for annotation.

We will first calculate these scores for the unlabeled pool, then compare which messages each strategy would select.



### 3.2 Implementing Acquisition Functions

Let $p(y \mid x)$ denote the classifier's predicted probability for intent $(y)$ given message $(x)$.

**Least confidence**

$$
a_{\mathrm{LC}}(x) = 1 - \max_y p(y \mid x)
$$

**Margin sampling**

$$
a_{\mathrm{margin}}(x) = 1 - (p_{(1)} - p_{(2)})
$$

where $p_{(1)}$ and $p_{(2)}$ are the highest and second-highest predicted probabilities.

**Entropy sampling**

$$
a_{\mathrm{entropy}}(x) = -\sum_y p(y \mid x)\log p(y \mid x)
$$

Each function receives a matrix of predicted probabilities and returns one uncertainty score per message.

In [ ]:
import numpy as np

def least_confidence(probs):
    """Higher score = lower confidence."""
    return 1 - np.max(probs, axis=1)


def margin_sampling(probs):
    """Higher score = smaller difference between top two classes."""
    sorted_probs = np.sort(probs, axis=1)
    return 1 - (sorted_probs[:, -1] - sorted_probs[:, -2])


def entropy_sampling(probs):
    """Higher score = greater predictive entropy."""
    return -np.sum(probs * np.log(probs + 1e-12), axis=1)


In [ ]:

# Two hypothetical predictions over three classes
example_probs = np.array([
    [0.90, 0.05, 0.05],  # Confident prediction
    [0.40, 0.35, 0.25]   # Uncertain prediction
])

df_example_scores = pd.DataFrame({
    "least_confidence": least_confidence(example_probs),
    "margin": margin_sampling(example_probs),
    "entropy": entropy_sampling(example_probs)
}, index=["Confident", "Uncertain"])

display(df_example_scores.round(3))


**Important**: the acquisition scores are not probabilities. They also have different numerical ranges, so their absolute values should not be compared directly across methods.


### 3.3 Scoring the Unlabeled Pool

We can now calculate uncertainty scores for every message in the unlabeled pool ($U_0$).

The classifier has never been trained on these messages. We will use only their text and predicted probabilities to calculate acquisition scores.

The reference labels remain hidden from the acquisition process.


In [ ]:

# Predict probabilities for the entire unlabeled pool
pool_probs = model.predict_proba(df_pool["text"])

# Store predictions and acquisition scores
df_scores = df_pool[["text"]].copy()

df_scores["predicted_intent"] = model.predict(df_pool["text"])
df_scores["confidence"] = pool_probs.max(axis=1)

df_scores["least_confidence"] = least_confidence(pool_probs)
df_scores["margin"] = margin_sampling(pool_probs)
df_scores["entropy"] = entropy_sampling(pool_probs)

display(df_scores.head())



### 3.4 Comparing Selected Queries

Each acquisition function ranks messages according to a different interpretation of uncertainty.

Let's inspect the five highest-ranked messages for each strategy.

**Questions to consider:**

- Do the strategies select the same messages?
- Which predicted intents appear most frequently?
- Are the messages genuinely difficult to classify?
- Would a human annotator necessarily find these examples difficult?


In [ ]:
N_QUERIES = 5

for strategy in ["least_confidence", "margin", "entropy"]:
    print(f"\nTop {N_QUERIES} queries — {strategy}")

    top_queries = df_scores.nlargest(N_QUERIES, strategy)

    display(
        top_queries[
            ["text", "predicted_intent", "confidence", strategy]
        ].round(3)
    )


In [ ]:

# Compare the top 20 examples selected by each strategy
TOP_K = 20

strategies = ["least_confidence", "margin", "entropy"]

selected_indices = {
    strategy: set(df_scores.nlargest(TOP_K, strategy).index)
    for strategy in strategies
}

# Count overlapping selections
overlap = pd.DataFrame(
    {
        s1: [
            len(selected_indices[s1] & selected_indices[s2])
            for s2 in strategies
        ]
        for s1 in strategies
    },
    index=strategies
)

display(overlap)



## 4. One Active-Learning Round

### 4.1 Selecting a Query Batch

So far, we have ranked the messages in our unlabeled pool according to their uncertainty. Now we will select a batch of examples for annotation.

For this first round, we will use **least confidence sampling** and select the 20 most uncertain messages.

The selected examples form our query batch:

$$
Q_0 = \operatorname{TopK}_{x \in U_0} a_0(x)
$$

where $a_0(x)$ is the least-confidence acquisition score and $k=20$.

At this stage, we are selecting **messages**, not their labels. The labels will be obtained in the next step.


In [ ]:
# Select the most uncertain messages
query_indices = df_scores.nlargest(BATCH_SIZE, "least_confidence").index

# Query batch: only information available to the learner
df_query = df_scores.loc[
    query_indices,
    ["text", "predicted_intent", "confidence", "least_confidence"]
].copy()

print("Selected examples:", len(df_query))
display(df_query.head(10))


### 4.2 Simulating Human Annotation

In a real active-learning system, the selected messages would now be sent to human annotators. In our simiulations we will use an oracle.

Our simulated **oracle** receives the selected query batch $Q_0$ and returns the corresponding labels.

These labels were not available to the acquisition strategy when it selected the messages.

We can now compare the model's original predictions with the labels returned by the oracle.

In [ ]:
# Simulated oracle: reveal the labels of selected examples
df_annotated = df_pool.loc[query_indices, ["text", "intent"]].copy()

# Compare model predictions with the newly revealed labels
df_annotated["predicted_intent"] = df_query["predicted_intent"]

df_annotated["correct_prediction"] = (
    df_annotated["intent"] == df_annotated["predicted_intent"]
)

display(df_annotated.head(10))

print(
    "Correct predictions in queried batch:",
    df_annotated["correct_prediction"].sum(),
    "/",
    len(df_annotated)
)

### 4.3 Updating the Datasets

After annotation, the queried examples become part of the labeled training data.

We update both datasets:

$$
L_1 = L_0 \cup \operatorname{Label}(Q_0)
$$

$$
U_1 = U_0 \setminus Q_0
$$

The labeled set grows by 20 examples, while the unlabeled pool shrinks by the same amount.

Each queried example must be removed from the pool so that it cannot be selected again in a later round.


In [ ]:
# Add newly annotated examples to the labeled set
df_labeled_1 = pd.concat([
    df_labeled,
    df_annotated[["text", "intent"]]
])

# Remove queried examples from the unlabeled pool
df_pool_1 = df_pool.drop(query_indices).copy()

print("Before annotation:")
print("  Labeled:", len(df_labeled))
print("  Pool:", len(df_pool))

print("\nAfter annotation:")
print("  Labeled:", len(df_labeled_1))
print("  Pool:", len(df_pool_1))

# Verify that no queried examples remain in the pool
assert not df_labeled_1.index.intersection(df_pool_1.index).size
assert len(df_labeled_1) == len(df_labeled) + BATCH_SIZE
assert len(df_pool_1) == len(df_pool) - BATCH_SIZE



### 4.4 Retraining the Classifier

We now have additional labeled examples. The next step is to retrain the classifier using the expanded labeled set $L_1$.

This produces a new model $f_1$.

We will evaluate both $f_0$ and $f_1$ on the same validation set to see whether the additional annotations improved performance.

Note that improvement is **not guaranteed** after a single acquisition round. Active learning aims to improve label efficiency over multiple rounds, not necessarily after every individual batch.


In [ ]:

from sklearn.base import clone

# Create a fresh classifier with the same configuration
model_1 = clone(model)

# Train on the expanded labeled set
model_1.fit(df_labeled_1["text"], df_labeled_1["intent"])

# Evaluate on the validation set
y_val_pred_1 = model_1.predict(df_val["text"])

accuracy_1 = accuracy_score(df_val["intent"], y_val_pred_1)
f1_1 = f1_score(df_val["intent"], y_val_pred_1, average="macro")

# Compare before and after annotation
df_comparison = pd.DataFrame({
    "Model": ["Initial model (f0)", "After one round (f1)"],
    "Labeled examples": [len(df_labeled), len(df_labeled_1)],
    "Accuracy": [initial_accuracy, accuracy_1],
    "Macro-F1": [initial_f1, f1_1]
})

display(df_comparison.round(3))


print(f"Change in accuracy: {accuracy_1 - initial_accuracy:+.3f}")
print(f"Change in macro-F1: {f1_1 - initial_f1:+.3f}")


### 4.5 Interpreting the Result

We have now completed one active-learning round:

**Train -> Score -> Query -> Annotate -> Update -> Retrain**

**Questions to consider:**

1. Did validation performance improve after retraining?
2. How might we explain the observed change in performance?

**Key observations:**

- **If performance improved:** We cannot yet conclude that uncertainty sampling was responsible. Selecting 20 random examples might have produced a similar-or even greater-improvement.

- **If performance decreased:** Several explanations are possible:
  - Uncertainty sampling prioritizes difficult examples, which may not be representative of the overall data distribution.
  - With very little initial training data, the model may be unstable and sensitive to newly acquired examples.
  - Some selected messages may be inherently ambiguous, so their labels may provide limited learning value.

**Important:** A single round is not enough to judge an acquisition strategy. We need to compare against random sampling and evaluate performance over multiple rounds. The question is not whether every batch improves performance, but **whether a strategy achieves better performance with the same annotation budget**.



## 5. Comparing Active-Learning Strategies

### 5.1 Multiple rounds

Evaluating a single round is not enough to determine whether an acquisition strategy is effective.
We will now repeat the process (above) over several rounds and compare four strategies:

- **Random sampling:** Select examples randomly from the unlabeled pool.
- **Least confidence:** Select examples with the lowest maximum predicted probability.
- **Margin sampling:** Select examples with the smallest difference between the two most likely classes.
- **Entropy sampling:** Select examples with the highest predictive entropy.

All strategies will start with the **same initial labeled set**, use the **same annotation budget**, and be evaluated on the **same validation set**.

We will plot a **learning curve** showing how validation performance changes as more labels are acquired.

Our main question is:

**Which strategy achieves better classification performance for the same number of labeled examples?**


### 5.2 Implementing an Acquisition Strategy
We already implemented the uncertainty functions in Section 3. Rather than duplicating them, we can create a small function that selects the next batch using one of the four strategies.

In [ ]:

def select_queries(model, df_pool, strategy, batch_size, rng):
    """Select a batch of examples from the unlabeled pool."""

    # Random sampling does not require model predictions
    if strategy == "random":
        return rng.choice(
            df_pool.index.to_numpy(),
            size=batch_size,
            replace=False
        )

    # Predict probabilities for unlabeled examples
    probs = model.predict_proba(df_pool["text"])

    # Calculate acquisition scores
    if strategy == "least_confidence":
        scores = least_confidence(probs)
    elif strategy == "margin":
        scores = margin_sampling(probs)
    elif strategy == "entropy":
        scores = entropy_sampling(probs)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # Select the highest-scoring examples
    selected_positions = np.argsort(-scores, kind="stable")[:batch_size]

    return df_pool.index.to_numpy()[selected_positions]



### 5.3 Running the Active-Learning Experiment

We will implement the complete active-learning loop.

For each strategy, we start again with the same ($L_0$) and ($U_0$). At each round:

1. Select 20 examples from the unlabeled pool.
2. Reveal their reference labels (simulated annotation).
3. Add the examples to the labeled set and remove them from the pool.
4. Retrain the classifier.
5. Evaluate its performance on the validation set.

We record performance **before the first query and after every acquisition round**.

This allows us to compare strategies throughout the annotation process rather than relying on a single result.

![Active-learning experiment](img/al-experiment.svg)

In [ ]:

from sklearn.base import clone

def run_active_learning(strategy, seed=SEED):
    """Run an active-learning experiment for one acquisition strategy."""

    # instatiate a random number generator
    rng = np.random.default_rng(seed)

    # Each strategy starts from the same initial datasets
    labeled = df_labeled.copy()
    pool = df_pool.copy()

    # Train the initial classifier
    clf = clone(model)  # create a fresh, unfitted model with the same configuration
    clf.fit(labeled["text"], labeled["intent"])

    results = []

    for round_num in range(N_ROUNDS + 1):

        # Evaluate the current model
        y_pred = clf.predict(df_val["text"])

        results.append({
            "strategy": strategy,
            "round": round_num,
            "n_labeled": len(labeled),
            "accuracy": accuracy_score(df_val["intent"], y_pred),
            "macro_f1": f1_score(
                df_val["intent"], y_pred, average="macro"
            )
        })

        # Stop after evaluating the final round
        if round_num == N_ROUNDS:
            break

        # Select the next query batch
        query_indices = select_queries(
            clf, pool, strategy, BATCH_SIZE, rng
        )

        # Simulated annotation: reveal selected labels
        newly_labeled = pool.loc[query_indices, ["text", "intent"]]

        # Update the labeled set and unlabeled pool
        labeled = pd.concat([labeled, newly_labeled])
        pool = pool.drop(query_indices)

        # Retrain using the expanded labeled set
        clf = clone(model)
        clf.fit(labeled["text"], labeled["intent"])

    return pd.DataFrame(results)



### 5.4 Comparing Learning Curves

A learning curve shows how model performance changes as the number of labeled training examples increases.

For our experiment:

- **X-axis:** Number of labeled examples.
- **Y-axis:** Validation macro-F1.
- **Each line:** A different acquisition strategy.

If an uncertainty-based strategy achieves higher macro-F1 than random sampling at the same labeling budget, this suggests that it is acquiring more useful training examples under these experimental conditions.

However, performance may fluctuate between rounds, and no strategy is guaranteed to outperform random sampling.


In [ ]:
strategies = [
    "random",
    "least_confidence",
    "margin",
    "entropy"
]

# Store the results of each experiment
all_results = []

# Run a separate active-learning experiment for each strategy
for strategy in strategies:

    print(f"Running active learning: {strategy}")

    # Start from the same initial labeled set and unlabeled pool
    # Run all acquisition rounds and record validation performance
    strategy_results = run_active_learning(strategy)

    # Collect the results for later comparison
    all_results.append(strategy_results)

# Combine results from all four strategies
df_results = pd.concat(all_results, ignore_index=True)

display(df_results)

In [ ]:
import matplotlib.pyplot as plt

plt_one_run = plt.figure(figsize=(9, 5))

for strategy in strategies:
    subset = df_results[df_results["strategy"] == strategy]

    plt.plot(
        subset["n_labeled"],
        subset["macro_f1"],
        marker="o",
        label=strategy
    )

plt.xlabel("Number of labeled examples")
plt.ylabel("Validation macro-F1")
plt.title("Active Learning: Comparing Acquisition Strategies")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.5 Interpreting the Results

Examine the learning curves and final performance table.

**Questions to consider:**

1. Which acquisition strategy achieves the highest validation macro-F1 after using the full annotation budget?
2. Do the uncertainty-based strategies consistently outperform random sampling?
3. Does performance improve after every acquisition round?
4. Do least confidence, margin, and entropy behave similarly? Why might this be?
5. Would your conclusion change if you considered performance throughout the learning curve rather than only at the final round?

**Key takeaway:** Active learning should be evaluated in terms of **label efficiency**, not simply whether performance increases after acquiring more data.

A useful strategy should help us achieve better performance with the same annotation budget—or comparable performance with fewer labels.



In [ ]:

# Compare strategies after using the full annotation budget
df_final = (
    df_results[df_results["round"] == N_ROUNDS]
    [["strategy", "n_labeled", "accuracy", "macro_f1"]]
    .sort_values("macro_f1", ascending=False)
)

display(df_final.round(3))


**Questions:**

1. Which pairs of intents are most frequently confused?
2. Are the errors concentrated in a few classes or spread across many?
3. Do the confused intents have something in common?

The confusion matrix tells us **where mistakes occur**, but not necessarily **what causes them**.


## 6. Evaluating Variability

### 6.1 Why Repeat Active-Learning Experiments?

Our previous comparison used one initial labeled set and one sequence of acquisitions per strategy.

However, active-learning results may depend on:

- **Initial labeled examples:** Different starting examples can produce different models and uncertainty estimates.
- **Selected queries:** Small differences in predictions can change which examples are acquired.
- **Random sampling:** Different random batches may lead to different learning curves.

To assess how reliable our conclusions are, we will repeat the experiment using different random seeds.

For each repetition, all acquisition strategies will share the same initial labeled set and evaluation data. We will then compare their **average performance and variability**.

A strategy that performs well in one run may not necessarily perform well consistently.

### 6.2 Allowing Different Initial Labeled Sets
So far, every active-learning experiment has started with the same initial labeled set (`df_labeled`) and unlabeled pool (`df_pool`).

Although `run_active_learning()` accepts a random seed, this seed only controls randomness during query selection. It does **not** change which examples are initially labeled.

To repeat the experiment with different starting conditions, we need to allow the function to accept alternative initial datasets.

**We only need to change two parts of the function.**

**1. Extend the function signature** with two optional parameters:

```python
def run_active_learning(strategy, seed=SEED,
                        initial_labeled=None, initial_pool=None):
```

**2. Replace the fixed initialization** of the labeled set and pool.

Previously:

```python
labeled = df_labeled.copy()
pool = df_pool.copy()
```

Now:

```python
labeled = (
    initial_labeled.copy()
    if initial_labeled is not None
    else df_labeled.copy()
)

pool = (
    initial_pool.copy()
    if initial_pool is not None
    else df_pool.copy()
)
```

When alternative datasets are provided, the function uses them. Otherwise, it behaves exactly as before.

In [ ]:
def run_active_learning(strategy, seed=SEED,
                        initial_labeled=None, initial_pool=None):
    """Run an active-learning experiment for one strategy."""

    rng = np.random.default_rng(seed)

    # NEW: Allow alternative initial datasets
    # Fall back to the original datasets if none are provided
    labeled = (
        initial_labeled.copy()
        if initial_labeled is not None
        else df_labeled.copy()
    )
    pool = (
        initial_pool.copy()
        if initial_pool is not None
        else df_pool.copy()
    )

    clf = clone(model)
    clf.fit(labeled["text"], labeled["intent"])

    results = []

    for round_num in range(N_ROUNDS + 1):
        y_pred = clf.predict(df_val["text"])

        results.append({
            "strategy": strategy,
            "round": round_num,
            "n_labeled": len(labeled),
            "accuracy": accuracy_score(df_val["intent"], y_pred),
            "macro_f1": f1_score(
                df_val["intent"], y_pred, average="macro"
            )
        })

        if round_num == N_ROUNDS:
            break

        query_indices = select_queries(
            clf, pool, strategy, BATCH_SIZE, rng
        )

        # Simulated oracle
        newly_labeled = pool.loc[query_indices, ["text", "intent"]]

        labeled = pd.concat([labeled, newly_labeled])
        pool = pool.drop(query_indices)

        clf = clone(model)
        clf.fit(labeled["text"], labeled["intent"])

    return pd.DataFrame(results)



### 6.3 Repeating the Experiment

We will repeat our comparison using **five** different initial labeled sets.

Then, as before, for each repetition:

1. Sample 10 initial labeled examples per intent.
2. Use the remaining training candidates as the unlabeled pool.
3. Run all four acquisition strategies with the same starting data.
4. Record validation performance after each round.

This gives us multiple learning curves for each strategy.

We can then examine whether the differences between strategies persist across repetitions.

In [ ]:
# number of repetitions
N_REPEATS = 5

all_results = []

for repeat in range(N_REPEATS):
    seed = SEED + repeat

    # Create a new initial labeled set
    initial_labeled = (
        df_train_candidates
        .groupby("intent", group_keys=False)
        .sample(n=N_INITIAL_PER_CLASS, random_state=seed)
        .copy()
    )

    initial_pool = df_train_candidates.drop(initial_labeled.index).copy()

    # Compare all strategies from the same starting conditions
    for strategy in strategies:
        result = run_active_learning(
            strategy,
            seed=seed,
            initial_labeled=initial_labeled,
            initial_pool=initial_pool
        )

        result["repeat"] = repeat
        all_results.append(result)

df_repeated = pd.concat(all_results, ignore_index=True)

display(df_repeated.head())



### 6.4 Mean Learning Curves and Variability

Instead of plotting every individual run, we will summarize the results using:

- **Mean macro-F1:** Average validation performance across repetitions.
- **Standard deviation:** How much performance varies between runs.

The shaded region in the plot represents **mean ± one standard deviation**.

A narrow band indicates that the observed performance is relatively consistent across repetitions. A wider band indicates greater variability.

These bands describe variation between runs; they are not confidence intervals.


In [ ]:
# let's prepare the data summary for the plot
df_summary = (
    df_repeated
    .groupby(["strategy", "n_labeled"])["macro_f1"]
    .agg(["mean", "std"])
    .reset_index()
)

# let's inpect some rows
display(df_summary.head())


In [ ]:
plt_multiple_runs = plt.figure(figsize=(9, 5))

# Plot the results for each acquisition strategy
for strategy in strategies:
    subset = df_summary[df_summary["strategy"] == strategy]

    # Get the number of labels, mean performance, and standard deviation
    x = subset["n_labeled"].to_numpy()
    mean = subset["mean"].to_numpy()
    std = subset["std"].to_numpy()

    # Plot the mean learning curve
    plt.plot(x, mean, marker="o", label=strategy)

    # Shade the region within one standard deviation of the mean
    plt.fill_between(
        x,
        mean - std,
        mean + std,
        alpha=0.15
    )

# Format the plot
plt.xlabel("Number of labeled examples")
plt.ylabel("Validation macro-F1")
plt.title("Active Learning: Mean Performance Across Repetitions")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 6.5 Interpreting the Results

Compare the repeated experiments with the learning curves from Section 5.

**Questions to consider:**

1. Does the strategy that performed best in the original experiment still perform best on average?
2. How much does performance vary between repetitions?
3. Are the differences between acquisition strategies large compared with the observed variability?
4. Is any strategy consistently better than random sampling throughout the annotation process?

**Key takeaway:** A single learning curve can be misleading. Active-learning strategies should be compared under equivalent conditions and across multiple runs.

Even repeated experiments do not guarantee that one strategy will generalize to every dataset or classifier. Their effectiveness depends on the task, model, data distribution, and annotation process.


In [ ]:
display(plt_one_run)
display(plt_multiple_runs)

## 7. Applying Active Learning to Your Project

Now it's your turn! Extend your semester project with an **active-learning workflow**, using this notebook as a reference.

Your goal is to investigate whether active learning can reduce the amount of labeled data required to achieve good performance on your task.

The methods demonstrated in this notebook focus on multiclass classification. Your project may involve a different task or model, so you will need to **adapt the acquisition function to your setting**.

### 7.1 Design Your Active-Learning Scenario

**1. Start from your baseline.**

Reuse your existing dataset, preprocessing, model, and evaluation metrics. Your active-learning experiment should build on the task you have already defined.

**2. Define your initial labeled set.**

- Select a small initial labeled set from your training data.
- As a starting point, consider approximately **5–10% of the training pool**, adjusting this to your dataset size and model requirements.
- Ensure the initial set is sufficient to train a meaningful model. For classification, it should contain examples of all known classes.
- Explain how you selected the initial examples and justify the size of the set.
- Reserve the remaining training examples as the unlabeled pool.

There is no universally optimal initial set size. Your choice should balance having enough labeled examples to start learning against preserving a substantial pool for active learning.

**3. Simulate human annotation.**

- Use the existing reference labels or verified target outputs as a **simulated annotation oracle**, as demonstrated in this notebook.
- Reveal these annotations only when examples are selected for acquisition.
- Keep validation and test data separate from the acquisition process.

### 7.2 Select and Implement Acquisition Functions

**1. Define what uncertainty means for your task.**

Consider the type of predictions your model produces:

- **Multiclass classification:** You may use least confidence, margin sampling, or entropy.
- **Binary classification:** You may use distance from 0.5, which gives the same ranking as the standard uncertainty measures.
- **Generative models / structured information extraction:** Standard classification-based measures may not apply directly. Consider an alternative, such as disagreement between multiple generated outputs.

**2. Choose and justify your acquisition function.**

Explain:

- What model output or signal is used to estimate uncertainty?
- How is this converted into an acquisition score?
- Why is this measure appropriate for your task?
- What are its limitations?

You do not need to implement every acquisition function covered in the lecture. Select at least **one uncertainty-based acquisition function** that is appropriate for your project.

If your model does not provide class probabilities, propose a reasonable alternative and explain your choice.

**3. Implement and compare acquisition strategies.**

Implement at least two conditions:

- **Random sampling:** Select examples randomly from the unlabeled pool.
- **Uncertainty sampling:** Select examples according to your chosen acquisition function.

You may investigate additional acquisition functions if they are relevant to your project.

Use the same initial labeled set, acquisition batch size, total annotation budget, and evaluation procedure for all conditions.

### 7.3 Evaluate Label Efficiency

Run multiple acquisition rounds, adding the selected examples to the labeled set and updating your model after each round.

- Evaluate model performance after each acquisition round.
- Plot learning curves showing performance against the number of labeled examples.
- Repeat the experiment using different random seeds or initial samples.
- Report average performance and variability across repetitions.

Use evaluation metrics appropriate to your task, such as accuracy, macro-F1, positive-class F1, or extraction F1.

### 7.4 Analyze Your Results

Discuss the following questions:

1. **Initial set:** How did you choose its size? Was it sufficient to train a useful initial model?
2. **Acquisition function:** How did you define uncertainty for your task, and why?
3. **Performance:** Does uncertainty sampling outperform random sampling?
4. **Label efficiency:** Does the advantage change as more examples are labeled?
5. **Stability:** How consistent are the results across repetitions?
6. **Limitations:** Are there cases where your acquisition function may fail to identify useful examples?
7. **Project relevance:** Based on your results, is active learning a promising approach for your project?

Remember that **improved performance after acquiring additional labels does not, by itself, demonstrate that active learning is beneficial**. The important comparison is whether strategic selection performs better than random selection under the same annotation budget.

### 7.5 Documenting Your Experiment

Make an effort to document your experiments. Your notebook should **tell the story of your experiment**, not just contain code and results. A reader should be able to understand what you investigated, why you made particular decisions, what you observed, and what those observations mean.

In particular:

- **Explain your decisions.** Document how you selected the initial labeled set, defined uncertainty for your task, and chose your acquisition function.
- **Keep your experiments reproducible.** Record the relevant parameters, random seeds, data splits, annotation budgets, and evaluation settings. Make sure the notebook can be executed again to reproduce your results.
- **Present results clearly.** Use appropriately labeled plots or tables, and explain what they show rather than leaving the reader to interpret them.
- **Interpret your findings.** Discuss whether uncertainty sampling outperformed random sampling, how results changed with the labeling budget, and what limitations you observed.
- **Keep track of open questions.** Note assumptions, unexpected behavior, and decisions you may need to revisit in later labs.

**Think of your notebook as an evolving experimental report:** code demonstrates what you did, while explanations and results communicate what you learned.